# P-Task2: Student Result Processor

**Problem:** department ke paas students ki CSV files ka folder hai. Kuch files corrupt hain, kuch missing hain
aur kuch mein invalid data hai. Mujhe aisa system banana hai jo saari files safely padhe,
failed students flag kare aur clean results file likhe, **bina crash hue**.

**Concepts used:** `open()` + `with`, `csv` module, `try / except / else / finally`,
`FileNotFoundError`, `ValueError`, custom exception, `os` module.

## 1. Test data banana
Asli files mere paas nahi thin, isliye khud sample files bana rahi hoon (ek theek, ek invalid rows wali,
ek corrupt, ek empty). Ek file ka naam list mein hai magar file bani hi nahi (missing).

In [1]:
import csv
import os

folder = "student_files"
os.makedirs(folder, exist_ok=True)

good = """roll_no,name,subject,marks
CS-01,Ayesha Khan,Python,82
CS-02,Bilal Ahmed,Python,35
CS-03,Fatima Noor,Python,48
CS-04,Usman Tariq,Python,91
"""

some_bad = """roll_no,name,subject,marks
SE-01,Zainab Raza,Python,74
SE-02,Omar Farooq,Python,abc
SE-03,,Python,60
SE-04,Daniyal Khan,Python,150
SE-05,Kamran Yousaf,Python,38
"""

corrupt = b"\x89PNG\r\n\x1a\n\x00\xff\xfe garbage bytes yahan hain"

with open(os.path.join(folder, "cs_batch.csv"), "w", newline="") as f:
    f.write(good)
with open(os.path.join(folder, "se_batch.csv"), "w", newline="") as f:
    f.write(some_bad)
with open(os.path.join(folder, "corrupt_batch.csv"), "wb") as f:
    f.write(corrupt)
open(os.path.join(folder, "empty_batch.csv"), "w").close()

file_list = ["cs_batch.csv", "se_batch.csv", "corrupt_batch.csv", "empty_batch.csv", "ee_batch.csv"]
os.listdir(folder)

['corrupt_batch.csv', 'se_batch.csv', 'cs_batch.csv', 'empty_batch.csv']

## 2. Custom exceptions

In [2]:
class ResultError(Exception):
    """Base class, hamare saare errors isi se bante hain."""


class InvalidRecordError(ResultError):
    """Ek row ka data galat hai."""


class BadFileError(ResultError):
    """Poori file kharab hai (header nahi, padhi nahi ja rahi)."""

## 3. Ek row check karna
Rules: roll_no aur name khali nahi hone chahiye, marks number hon aur 0 se 100 ke beech hon.
Pass marks = 40.

In [3]:
PASS_MARKS = 40


def check_row(row):
    roll_no = (row.get("roll_no") or "").strip()
    name = (row.get("name") or "").strip()
    if roll_no == "" or name == "":
        raise InvalidRecordError("roll_no ya name khali hai")
    try:
        marks = float(row["marks"])
    except (ValueError, TypeError):
        raise InvalidRecordError(f"marks number nahi hain: {row.get('marks')!r}")
    if not 0 <= marks <= 100:
        raise InvalidRecordError(f"marks 0-100 se bahar hain: {marks}")
    return {"roll_no": roll_no, "name": name, "subject": row["subject"].strip(), "marks": marks}

## 4. File safely padhna
`try / except / else / finally` ka poora use:
* `except` - file ke masle pakadta hai
* `else` - sirf tab chalta hai jab koi error na aaye
* `finally` - har haal mein chalta hai (yahan files ka count)

In [4]:
all_students = []
problems = []          # (file, line, reason)
files_checked = 0


def read_file(filename):
    global files_checked
    path = os.path.join(folder, filename)
    students = []
    try:
        with open(path, newline="", encoding="utf-8") as f:
            reader = csv.DictReader(f)
            if reader.fieldnames is None or "marks" not in reader.fieldnames:
                raise BadFileError("header row nahi mili")
            for line_no, row in enumerate(reader, start=2):
                try:
                    students.append(check_row(row))
                except InvalidRecordError as e:
                    problems.append((filename, line_no, str(e)))
    except FileNotFoundError:
        problems.append((filename, 0, "file missing hai"))
    except UnicodeDecodeError:
        problems.append((filename, 0, "file corrupt hai (padhi nahi ja saki)"))
    except BadFileError as e:
        problems.append((filename, 0, str(e)))
    else:
        print(f"{filename}: OK, {len(students)} valid students")
    finally:
        files_checked += 1
    return students


for name in file_list:
    all_students.extend(read_file(name))

print("\nFiles checked:", files_checked)
print("Valid students:", len(all_students))

cs_batch.csv: OK, 4 valid students
se_batch.csv: OK, 2 valid students

Files checked: 5
Valid students: 6


## 5. Problems ki list

In [5]:
for filename, line_no, reason in problems:
    where = f"line {line_no}" if line_no else "poori file"
    print(f"{filename:<18} {where:<11} {reason}")

se_batch.csv       line 3      marks number nahi hain: 'abc'
se_batch.csv       line 4      roll_no ya name khali hai
se_batch.csv       line 5      marks 0-100 se bahar hain: 150.0
corrupt_batch.csv  poori file  file corrupt hai (padhi nahi ja saki)
empty_batch.csv    poori file  header row nahi mili
ee_batch.csv       poori file  file missing hai


## 6. Failed students flag karna

In [6]:
failed = [s for s in all_students if s["marks"] < PASS_MARKS]
print("Failed students:")
for s in failed:
    print(f"  {s['roll_no']}  {s['name']:<15} {s['marks']}")

Failed students:
  CS-02  Bilal Ahmed     35.0
  SE-05  Kamran Yousaf   38.0


## 7. Clean results file likhna
Sab valid students `clean_results.csv` mein jayenge, status ke saath.
Pehle purani file ho to `os.remove()` se hata di.

In [7]:
out_file = os.path.join(folder, "clean_results.csv")
if os.path.exists(out_file):
    os.remove(out_file)

with open(out_file, "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["roll_no", "name", "subject", "marks", "status"])
    for s in sorted(all_students, key=lambda s: s["roll_no"]):
        status = "FAIL" if s["marks"] < PASS_MARKS else "PASS"
        writer.writerow([s["roll_no"], s["name"], s["subject"], s["marks"], status])

with open(out_file) as f:
    print(f.read())

roll_no,name,subject,marks,status
CS-01,Ayesha Khan,Python,82.0,PASS
CS-02,Bilal Ahmed,Python,35.0,FAIL
CS-03,Fatima Noor,Python,48.0,PASS
CS-04,Usman Tariq,Python,91.0,PASS
SE-01,Zainab Raza,Python,74.0,PASS
SE-05,Kamran Yousaf,Python,38.0,FAIL



## Conclusion
5 files mein se 2 sahi nikli, 1 corrupt, 1 empty aur 1 missing thi, phir bhi program kahin crash nahi hua.
Har masla `problems` list mein record ho gaya aur valid students ka clean result file ban gaya.